In [1]:
# Colab setup: mount Drive and move into the project folder (edit PROJECT if your folder is elsewhere)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/Colab Notebooks/careloop-no-show-guard"
    os.chdir(PROJECT)
except ImportError:
    pass                                   # not in Colab: stay in the current folder
print(os.getcwd()); print(os.listdir("."))

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/careloop-no-show-guard
['02_queries.sql', 'data', 'test_risk_scores.csv', '03_model.ipynb', 'notes_analysed.csv', '04_genai.ipynb', 'rejected_rows.json', 'clinics_enriched.csv', 'load_data.sql', 'appointments_clean.csv', '01_ingest_clean.ipynb', '02_run_queries_colab.ipynb']


# 02 – Run the Part 2 queries in Colab (SQLite)
Runs `02_queries.sql` against `load_data.sql`. Requires notebook 01 to have run first. The submitted SQL is the `.sql` file; MySQL Workbench output is the reference if you have it.

In [2]:
import sqlite3, re, pandas as pd
pd.set_option("display.width", 200)

con = sqlite3.connect(":memory:")              # fresh database every run
raw = open("02_queries.sql").read()

# 1) remove comments and the two MySQL-only lines (SQLite has no CREATE DATABASE / USE)
sql = re.sub(r"--[^\n]*", "", raw)             # removes whole-line and end-of-line comments
sql = re.sub(r"(?im)^\s*(CREATE DATABASE|USE)\b[^;]*;", "", sql)

# 2) split into statements
statements = [s.strip() for s in sql.split(";") if s.strip()]

loaded = False
for st in statements:
    if st.upper().startswith("CREATE TABLE"):
        con.execute(st)
        continue
    if not loaded:                              # tables exist -> run load_data.sql once
        con.executescript(open("load_data.sql").read())
        con.commit()
        loaded = True
    print("\n" + "=" * 70 + "\n" + st[:90].replace("\n", " ") + " ...")
    try:
        if st.upper().startswith(("SELECT", "WITH")):
            print(pd.read_sql_query(st, con).to_string(index=False))
        else:
            con.execute(st)
    except sqlite3.IntegrityError as e:         # the duplicate A0001 insert lands here
        print("ERROR (copy this into the comment in 02_queries.sql):", e)


SELECT COUNT(*) AS clinics_rows FROM clinics ...
 clinics_rows
            8

SELECT COUNT(*) AS appointments_rows, SUM(no_show) AS no_shows FROM appointments ...
 appointments_rows  no_shows
               990       218

INSERT INTO appointments VALUES ('A0001','P999','C01','2025-01-01','2025-01-05',30,'F',5.0 ...
ERROR (copy this into the comment in 02_queries.sql): UNIQUE constraint failed: appointments.appointment_id

SELECT c.clinic_name, c.city, COUNT(*) AS appointments, SUM(a.no_show) AS no_shows,        ...
             clinic_name   city  appointments  no_shows  no_show_pct
    Hadapsar Care Centre   Pune           143        39         27.3
    Malviya Nagar Clinic Jaipur           142        38         26.8
   Kothrud Family Clinic   Pune           160        36         22.5
      Vijay Nagar Clinic Indore           101        21         20.8
Edappally Medical Centre  Kochi           142        23         16.2
   Vaishali Nagar Clinic Jaipur           130        18         